In [1]:
# ═══════════════════════════════════════════════════════════════
# CELL 1: Setup & Imports
# ═══════════════════════════════════════════════════════════════
import sys, os, time, warnings, gc
warnings.filterwarnings('ignore')

possible_src_paths = [
    '/home/jovyan/work/src',
    os.path.abspath('../src'),
    os.path.abspath('./src')
]
for path in possible_src_paths:
    if os.path.exists(path):
        if path not in sys.path:
            sys.path.insert(0, path)
        print(f"✅ Loaded src from: {path}")
        break

from config.spark_config import create_spark_session, silver_path
from config.minio_config import ensure_buckets_exist

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, FloatType, LongType,
    IntegerType, BooleanType, ShortType, TimestampType
)
import pyarrow as pa 
import pyarrow.parquet as pq 
print("✅ Imports OK")

✅ Loaded src from: /home/jovyan/work/src
✅ Imports OK


In [2]:
# ═══════════════════════════════════════════════════════════════
# CELL 2: Spark + MinIO (2 workers)
# ═══════════════════════════════════════════════════════════════
from config.spark_config import bronze_path
ensure_buckets_exist()
spark = create_spark_session('MapReduceToSilver')

sc = spark.sparkContext

print(f"Spark version:       {spark.version}")
print(f"Master:              {sc.master}")
print(f"Default parallelism: {sc.defaultParallelism}")
print(f"Silver path:         {silver_path()}")
print(f"Bronze path:         {bronze_path()}")


# Verify cluster
test_rdd = sc.parallelize(range(12), 12)
result = test_rdd.glom().map(len).collect()
print(f"\n🔍 Cluster test: {len(result)} partitions distributed")
print(f"   Partition sizes: {result}")
print(f"   → {'✅ 2 workers active' if len(result) >= 6 else '⚠️ Check worker connections'}")

Bucket exists: electronics-bronze
Bucket exists: electronics-silver
Bucket exists: electronics-gold
SparkSession created: MapReduceToSilver
MinIO endpoint: http://minio:9000
Workers: 2 × 3 cores | Driver: 2g | Executor: 2g
Spark version:       3.5.0
Master:              local[*]
Default parallelism: 12
Silver path:         s3a://electronics-silver
Bronze path:         s3a://electronics-bronze

🔍 Cluster test: 12 partitions distributed
   Partition sizes: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
   → ✅ 2 workers active


In [15]:
# ═══════════════════════════════════════════════════════════════
# CELL 3: Schema & Config
# ═══════════════════════════════════════════════════════════════

from pyspark.sql.types import ArrayType 
from config.spark_config import bronze_path


REVIEW_SCHEMA = StructType([
    StructField("rating",            FloatType(),   nullable=True),
    StructField("title",             StringType(),  nullable=True),
    StructField("text",              StringType(),  nullable=True),
    StructField("parent_asin",       StringType(),  nullable=False),
    StructField("user_id",           StringType(),  nullable=False),
    StructField("timestamp",         LongType(),    nullable=True),
    StructField("helpful_vote",      IntegerType(), nullable=True),
    StructField("verified_purchase", BooleanType(), nullable=True),
])

META_SCHEMA = StructType([
    StructField("main_category",   StringType(),   nullable=True),
    StructField("title",           StringType(),   nullable=True),
    StructField("average_rating",  FloatType(),    nullable=True),
    StructField("rating_number",   IntegerType(),  nullable=True),
    StructField("features",        StringType(),   nullable=True),  
    StructField("description",     StringType(),   nullable=True),  
    StructField("price",           StringType(),   nullable=True),
    StructField("store",           StringType(),   nullable=True),
    StructField("categories",      StringType(),   nullable=True),  
    StructField("parent_asin",     StringType(),   nullable=False),
    StructField("bought_together", StringType(),   nullable=True), 
])

REVIEW_COLUMNS = ['rating', 'title', 'text', 'parent_asin',
                  'user_id', 'timestamp', 'helpful_vote', 'verified_purchase']

META_COLUMNS = ['main_category', 'title', 'average_rating', 'rating_number',
                'features', 'description', 'price', 'store', 'categories',
                'parent_asin', 'bought_together']

# ── MapReduce Config ─────────────────────────────────────────
NUM_WORKERS = 2
CORES_PER_WORKER = 3
TOTAL_SLOTS = NUM_WORKERS * CORES_PER_WORKER  # 6
CHUNK_SIZE_MB = 64                              # 64MB per chunk file
CHUNK_RECORDS = 300_000                         # ~64MB at ~200 bytes/record compressed
                      

# Paths
BRONZE_REVIEWS_PATH = bronze_path('reviews_chunks')
BRONZE_META_PATH = bronze_path('metadata_chunks')
SILVER_REVIEWS_PATH = silver_path('reviews')
SILVER_META_PATH = silver_path('metadata')

def _path_exists_and_not_empty(path: str) -> bool:
    """Check path tồn tại mà không scan data."""
    try:
        fs_path = spark._jvm.org.apache.hadoop.fs.Path(path)
        fs = fs_path.getFileSystem(spark._jsc.hadoopConfiguration())
        return fs.exists(fs_path) and len(fs.listStatus(fs_path)) > 0
    except:
        return False

print("✅ Config defined:")
print(f"   Workers: {NUM_WORKERS} × {CORES_PER_WORKER} cores = {TOTAL_SLOTS} slots")
print(f"   Chunk size: ~{CHUNK_SIZE_MB}MB (~{CHUNK_RECORDS:,} records/chunk)")
print(f"   Bronze chunks → {BRONZE_REVIEWS_PATH}")
print(f"   Silver output → {SILVER_REVIEWS_PATH}")
print(f"   Bronze meta chunks → {BRONZE_META_PATH}")
print(f"   Silver meta output → {SILVER_META_PATH}")

✅ Config defined:
   Workers: 2 × 3 cores = 6 slots
   Chunk size: ~64MB (~300,000 records/chunk)
   Bronze chunks → s3a://electronics-bronze/reviews_chunks
   Silver output → s3a://electronics-silver/reviews
   Bronze meta chunks → s3a://electronics-bronze/metadata_chunks
   Silver meta output → s3a://electronics-silver/metadata


In [16]:
# ═══════════════════════════════════════════════════════════════
# CELL 4: Download & Chunk functions
# ═══════════════════════════════════════════════════════════════
import io

import pandas as pd
from datasets import load_dataset
from config.minio_config import get_minio_client


def _pandas_pre_clean_reviews(pdf: pd.DataFrame) -> pd.DataFrame:
    cols = [c for c in REVIEW_COLUMNS if c in pdf.columns]
    pdf  = pdf[cols].copy()
    pdf.dropna(subset=['parent_asin', 'user_id'], inplace=True)
    pdf['rating']            = pd.to_numeric(pdf['rating'], errors='coerce').astype('float32')
    pdf['timestamp']         = pd.to_numeric(pdf['timestamp'], errors='coerce').astype('Int64')
    pdf['helpful_vote']      = pd.to_numeric(pdf['helpful_vote'], errors='coerce').fillna(0).astype('int32')
    pdf['verified_purchase'] = pdf['verified_purchase'].fillna(False).astype(bool)
    pdf['title']             = pdf['title'].fillna('')
    pdf['text']              = pdf['text'].fillna('')
    return pdf


def _pandas_pre_clean_meta(pdf: pd.DataFrame) -> pd.DataFrame:
    cols = [c for c in META_COLUMNS if c in pdf.columns]
    pdf  = pdf[cols].copy()
    pdf.dropna(subset=['parent_asin'], inplace=True)
    pdf['average_rating'] = pd.to_numeric(pdf['average_rating'], errors='coerce').fillna(0.0).astype('float32')
    pdf['rating_number']  = pd.to_numeric(pdf['rating_number'],  errors='coerce').fillna(0).astype('int32')

    LIST_COLS = ['features', 'description', 'categories', 'bought_together']
    for col in LIST_COLS:
        if col in pdf.columns:
            pdf[col] = pdf[col].apply(
                lambda x: ' | '.join(str(i) for i in x) if isinstance(x, list) else (str(x) if x is not None else '')
            )
    
    for col in ['title', 'store', 'price', 'main_category']:
        if col in pdf.columns:
            pdf[col] = pdf[col].fillna('').astype(str)
    
    return pdf


def _write_chunk_pyarrow(pdf: pd.DataFrame, bucket: str, obj_name: str, client) -> int:
    """
    Ghi chunk trực tiếp PyArrow → MinIO.
    Không tạo Spark job → tiết kiệm ~5-10s overhead/chunk.
    Returns: bytes written
    """
    table  = pa.Table.from_pandas(pdf, preserve_index=False)
    buffer = io.BytesIO()
    pq.write_table(table, buffer, compression='snappy')
    size = buffer.tell()
    buffer.seek(0)
    client.put_object(bucket, obj_name, buffer, size,
                      content_type='application/octet-stream')
    return size


def download_and_chunk(dataset_name: str, config_name: str,
                       columns: list, clean_fn, output_path: str,
                       bucket: str, chunk_records: int = CHUNK_RECORDS):
    """
    Download HuggingFace → PyArrow chunks → MinIO trực tiếp.
    Không qua Spark → không có job scheduling overhead.
    """
    print(f"\n{'='*60}")
    print(f"📥 DOWNLOAD & CHUNK: {config_name}")
    print(f"   Output: {output_path}  |  Chunk: ~{chunk_records:,} records")
    print(f"{'='*60}")

    client  = get_minio_client()
    ds      = load_dataset(dataset_name, config_name,
                           split="full", streaming=True, trust_remote_code=True)

    batch         = []
    chunk_idx     = 0
    total_records = 0
    total_bytes   = 0
    t_start       = time.time()

    for record in ds:
        batch.append({col: record.get(col) for col in columns})
        total_records += 1

        if len(batch) >= chunk_records:
            pdf = pd.DataFrame(batch)
            pdf = clean_fn(pdf)

            if len(pdf) > 0:
                # Path trong MinIO: "reviews_chunks/chunk_0000.parquet"
                prefix   = output_path.split(f"s3a://{bucket}/")[-1]
                obj_name = f"{prefix}/chunk_{chunk_idx:04d}.parquet"
                nb = _write_chunk_pyarrow(pdf, bucket, obj_name, client)
                total_bytes += nb

                elapsed = time.time() - t_start
                print(f"   📦 Chunk {chunk_idx:>4}: {len(pdf):>7,} rec | "
                      f"Total: {total_records:>10,} | "
                      f"{total_records/elapsed:,.0f} rec/s | "
                      f"{nb/1024**2:.1f}MB")

            # Giải phóng bộ nhớ explicit thay vì gc.collect()
            del pdf
            batch.clear()
            chunk_idx += 1

    # Chunk cuối
    if batch:
        pdf = pd.DataFrame(batch)
        pdf = clean_fn(pdf)
        if len(pdf) > 0:
            prefix   = output_path.split(f"s3a://{bucket}/")[-1]
            obj_name = f"{prefix}/chunk_{chunk_idx:04d}.parquet"
            nb = _write_chunk_pyarrow(pdf, bucket, obj_name, client)
            total_bytes += nb
            chunk_idx += 1
            print(f"   📦 Chunk final: {len(pdf):,} records | {nb/1024**2:.1f}MB")
        del pdf
        batch.clear()

    gc.collect()  # 1 lần duy nhất sau toàn bộ loop
    duration = (time.time() - t_start) / 60
    print(f"\n✅ Download hoàn tất:")
    print(f"   Records:  {total_records:>12,}")
    print(f"   Chunks:   {chunk_idx:>12}")
    print(f"   Size:     {total_bytes/1024**3:>11.2f} GB")
    print(f"   Duration: {duration:>11.1f} phút")
    return total_records, chunk_idx, duration


print("✅ Download & Chunk functions defined")

✅ Download & Chunk functions defined


In [5]:
# ═══════════════════════════════════════════════════════════════
# CELL 5: Chạy Download Reviews → Bronze Chunks
# ═══════════════════════════════════════════════════════════════
if _path_exists_and_not_empty(BRONZE_REVIEWS_PATH):
    print(f"⚠️  Bronze Review chunks đã tồn tại tại {BRONZE_REVIEWS_PATH}")
    print(f"    Để tải lại: xóa path rồi chạy lại cell này")
else:
    print("📦 Bronze Review chunks chưa có. Bắt đầu download...")
    download_and_chunk(
        dataset_name="McAuley-Lab/Amazon-Reviews-2023",
        config_name="raw_review_Electronics",
        columns=REVIEW_COLUMNS,
        clean_fn=_pandas_pre_clean_reviews,
        output_path=BRONZE_REVIEWS_PATH,
        bucket='electronics-bronze',
        chunk_records=CHUNK_RECORDS
    )

📦 Bronze Review chunks chưa có. Bắt đầu download...

📥 DOWNLOAD & CHUNK: raw_review_Electronics
   Output: s3a://electronics-bronze/reviews_chunks  |  Chunk: ~300,000 records
   📦 Chunk    0: 300,000 rec | Total:    300,000 | 8,564 rec/s | 66.8MB
   📦 Chunk    1: 300,000 rec | Total:    600,000 | 9,003 rec/s | 61.5MB
   📦 Chunk    2: 300,000 rec | Total:    900,000 | 9,066 rec/s | 63.3MB
   📦 Chunk    3: 300,000 rec | Total:  1,200,000 | 9,208 rec/s | 58.4MB
   📦 Chunk    4: 300,000 rec | Total:  1,500,000 | 9,158 rec/s | 63.3MB
   📦 Chunk    5: 300,000 rec | Total:  1,800,000 | 9,299 rec/s | 57.8MB
   📦 Chunk    6: 300,000 rec | Total:  2,100,000 | 9,331 rec/s | 60.7MB
   📦 Chunk    7: 300,000 rec | Total:  2,400,000 | 9,438 rec/s | 53.6MB
   📦 Chunk    8: 300,000 rec | Total:  2,700,000 | 9,522 rec/s | 53.3MB
   📦 Chunk    9: 300,000 rec | Total:  3,000,000 | 9,356 rec/s | 57.5MB
   📦 Chunk   10: 300,000 rec | Total:  3,300,000 | 9,404 rec/s | 57.5MB
   📦 Chunk   11: 300,000 rec | To

In [17]:
# ═══════════════════════════════════════════════════════════════
# CELL 6: Chạy Download Metadata → Bronze Chunks
# ═══════════════════════════════════════════════════════════════
if _path_exists_and_not_empty(BRONZE_META_PATH):
    print(f"⚠️  Bronze Meta chunks đã tồn tại tại {BRONZE_META_PATH}")
    print(f"    Để tải lại: xóa path rồi chạy lại cell này")
else:
    print("📦 Bronze Meta chunks chưa có. Bắt đầu download...")
    download_and_chunk(
        dataset_name="McAuley-Lab/Amazon-Reviews-2023",
        config_name="raw_meta_Electronics",
        columns=META_COLUMNS,
        clean_fn=_pandas_pre_clean_meta,
        output_path=BRONZE_META_PATH,
        bucket='electronics-bronze',
        chunk_records=CHUNK_RECORDS
    )

📦 Bronze Meta chunks chưa có. Bắt đầu download...

📥 DOWNLOAD & CHUNK: raw_meta_Electronics
   Output: s3a://electronics-bronze/metadata_chunks  |  Chunk: ~300,000 records
   📦 Chunk    0: 300,000 rec | Total:    300,000 | 569 rec/s | 212.2MB
   📦 Chunk    1: 300,000 rec | Total:    600,000 | 603 rec/s | 205.4MB
   📦 Chunk    2: 300,000 rec | Total:    900,000 | 623 rec/s | 197.1MB
   📦 Chunk    3: 300,000 rec | Total:  1,200,000 | 634 rec/s | 192.3MB
   📦 Chunk    4: 300,000 rec | Total:  1,500,000 | 648 rec/s | 190.3MB
   📦 Chunk final: 110,012 records | 68.3MB

✅ Download hoàn tất:
   Records:     1,610,012
   Chunks:              6
   Size:            1.04 GB
   Duration:        41.2 phút


In [18]:
# ═══════════════════════════════════════════════════════════════
# CELL 7: PHASE 2 — TRUE MapReduce: Bronze Chunks → Silver
# ═══════════════════════════════════════════════════════════════

def mapreduce_reviews_to_silver():
    print(f"🚀 MapReduce REVIEWS: Bronze Chunks → Silver")
   
    t_start = time.time()

    # ── STEP 1: Input Split ──────────────────────────────────
    print("\n📖 Step 1: Reading all Bronze chunks in parallel...")
    t1 = time.time()

    df_raw = (
        spark.read
        .schema(REVIEW_SCHEMA)
        .option('mergeSchema', 'false')
        .parquet(BRONZE_REVIEWS_PATH)
    )

    num_partitions = df_raw.rdd.getNumPartitions()
    print(f"   Chunks/Partitions: {num_partitions}")
    

    if num_partitions < TOTAL_SLOTS:
        df_raw = df_raw.repartition(TOTAL_SLOTS)
    elif num_partitions > TOTAL_SLOTS * 4:
        target = TOTAL_SLOTS * 3
        df_raw = df_raw.coalesce(target)

    # ── STEP 2: MAP Phase ────────────────────────────────────
    print("\n  Step 2: MAP — Parallel transform trên tất cả workers...")

    df_mapped = (
        df_raw
        .filter(F.col('parent_asin').isNotNull())
        .filter(F.col('user_id').isNotNull())
        .filter(F.col('timestamp').isNotNull())
        .filter(F.col('rating').between(1.0, 5.0))

        .withColumn('review_ts',
                    (F.col('timestamp') / 1000).cast(TimestampType()))
        .withColumn('review_year',
                    F.year('review_ts').cast(ShortType()))
        .withColumn('review_month',
                    F.month('review_ts').cast('byte'))

        .withColumn('title_clean',     F.lower(F.trim('title')))
        .withColumn('text_clean',      F.lower(F.trim('text')))
        .withColumn('text_word_count',
                    F.when(F.trim('text') == '', F.lit(0))
                     .otherwise(F.size(F.split(F.trim('text'), r'\s+'))))
        .withColumn('has_text', (F.length(F.trim('text')) > 0))

        .drop('timestamp', 'title', 'text')

        .select(
            'parent_asin', 'user_id', 'rating',
            'title_clean', 'text_clean', 'text_word_count',
            'review_ts', 'review_year', 'review_month',
            'helpful_vote', 'verified_purchase', 'has_text',
        )
        .filter(F.col('review_year').between(1996, 2024))
    )


    # ── STEP 3: REDUCE Phase ─────────────────────────────────
    print("\n🔀 Step 3: REDUCE — Global dedup + partition by time...")
    t3 = time.time()

    df_deduped  = df_mapped.dropDuplicates(['parent_asin', 'user_id', 'review_ts'])
   
    est_records      = num_partitions * 270_000
    records_per_file = (128 * 1024 * 1024) // 200
    num_output_files = max(TOTAL_SLOTS, est_records // records_per_file)
    print(f"   Est. records:  ~{est_records:,} | Output files: ~{num_output_files}")


    (
        df_mapped
        .dropDuplicates(['parent_asin', 'user_id', 'review_ts'])
        .repartition(int(num_output_files), 'review_year', 'review_month')
        .sortWithinPartitions('review_year', 'review_month', 'parent_asin')
        .write
        .mode('overwrite')
        .partitionBy('review_year', 'review_month')
        .option('compression', 'zstd')
        .parquet(SILVER_REVIEWS_PATH)
    )

    write_time = time.time() - t3
    total_time = time.time() - t_start

    # Đếm files thay vì scan data
    out_path = spark._jvm.org.apache.hadoop.fs.Path(SILVER_REVIEWS_PATH)
    out_fs   = out_path.getFileSystem(spark._jsc.hadoopConfiguration())
    num_out  = out_fs.listStatus(out_path).length

   
    print(f"🏁 REVIEWS HOÀN TẤT")
    print(f"   Write time:    {write_time:.0f}s")
    print(f"   Output dirs:   {num_out} partitions")
    print(f"   Total time:    {total_time/60:.1f} phút")
    return num_out
    


print("✅ MapReduce Review function defined")

✅ MapReduce Review function defined


In [19]:
# ═══════════════════════════════════════════════════════════════
# CELL 8: MapReduce Metadata → Silver
# ═══════════════════════════════════════════════════════════════

def mapreduce_metadata_to_silver():
    """
    MapReduce Metadata: Bronze Chunks → Silver.
    Metadata nhỏ (~800K records) nhưng vẫn dùng song song cho consistency.
    """
    print(f"🚀 MapReduce METADATA: Bronze Chunks → Silver")
   
    t_start = time.time()

    # ── INPUT SPLIT: Đọc tất cả chunks song song ──
    print("\n📖 Step 1: Reading metadata chunks...")
    df_raw = (
        spark.read
        .schema(META_SCHEMA)
        .option('mergeSchema', 'false')
        .parquet(BRONZE_META_PATH)
    )


    num_partitions = df_raw.rdd.getNumPartitions()
    print(f"   Partitions: {num_partitions} ")

    # Cân bằng partitions
    if num_partitions < TOTAL_SLOTS:
        df_raw = df_raw.repartition(TOTAL_SLOTS)

    # ── MAP PHASE: Transform song song ──
    print("\n  Step 2: MAP — Transform metadata...")

    df_mapped = (
        df_raw
        .filter(F.col('parent_asin').isNotNull())
        .filter(F.col('parent_asin') != '')

        # Parse price string → double
        .withColumn('price_parsed',
                    F.regexp_extract('price', r'\$?([\d,]+\.?\d*)', 1))
        .withColumn('price_numeric',
                    F.when(F.col('price_parsed') != '',
                           F.regexp_replace('price_parsed', ',', '').cast('double'))
                     .otherwise(F.lit(None).cast('double')))
        .drop('price_parsed')
        .withColumn('features',        F.coalesce(F.col('features'),        F.lit('')))
        .withColumn('description',     F.coalesce(F.col('description'),     F.lit('')))
        .withColumn('categories',      F.coalesce(F.col('categories'),      F.lit('')))
        .withColumn('bought_together', F.coalesce(F.col('bought_together'), F.lit('')))


        .select(
            'parent_asin',
            'main_category',
            'title',
            'average_rating',
            'rating_number',
            'price_numeric',
            F.col('price').alias('price_raw'),
            'store',
            'features',
            'description',
            'categories',
            'bought_together',
        )
    )

    # ── REDUCE PHASE: Global dedup trên parent_asin ──
    print("\n🔀 Step 3: REDUCE — Global dedup by parent_asin...")


    (
        df_mapped
        .dropDuplicates(['parent_asin'])
        .repartition(4, 'main_category')
        .sortWithinPartitions('main_category', 'parent_asin')
        .write
        .mode('overwrite')
        .option('compression', 'zstd')
        .parquet(SILVER_META_PATH)
    )

    total_time = time.time() - t_start

    print(f"\n🏁 METADATA HOÀN TẤT | Time: {total_time/60:.1f} phút")
    return True

print("✅ MapReduce Metadata function defined")

✅ MapReduce Metadata function defined


In [9]:
# ═══════════════════════════════════════════════════════════════
# CELL 9: Chạy MapReduce Reviews
# ═══════════════════════════════════════════════════════════════
if _path_exists_and_not_empty(SILVER_REVIEWS_PATH):
    print(f"⚠️  Silver Reviews đã tồn tại tại {SILVER_REVIEWS_PATH}")
    print(f"    Để chạy lại: mapreduce_reviews_to_silver()")
else:
    print("📦 Bắt đầu MapReduce Reviews...")
    mapreduce_reviews_to_silver()

📦 Bắt đầu MapReduce Reviews...
🚀 MapReduce REVIEWS: Bronze Chunks → Silver

📖 Step 1: Reading all Bronze chunks in parallel...
   Chunks/Partitions: 78

  Step 2: MAP — Parallel transform trên tất cả workers...

🔀 Step 3: REDUCE — Global dedup + partition by time...
   Est. records:  ~21,060,000 | Output files: ~31
🏁 REVIEWS HOÀN TẤT
   Write time:    1810s
   Output dirs:   <py4j.java_gateway.JavaMember object at 0x76abd9e4a850> partitions
   Total time:    30.3 phút


In [20]:
# ═══════════════════════════════════════════════════════════════
# CELL 10: Chạy MapReduce Metadata
# ═══════════════════════════════════════════════════════════════
if _path_exists_and_not_empty(SILVER_META_PATH):
    print(f"⚠️  Silver Metadata đã tồn tại tại {SILVER_META_PATH}")
    print(f"    Để chạy lại: mapreduce_metadata_to_silver()")
else:
    print("📦 Bắt đầu MapReduce Metadata...")
    mapreduce_metadata_to_silver()

📦 Bắt đầu MapReduce Metadata...
🚀 MapReduce METADATA: Bronze Chunks → Silver

📖 Step 1: Reading metadata chunks...
   Partitions: 13 

  Step 2: MAP — Transform metadata...

🔀 Step 3: REDUCE — Global dedup by parent_asin...

🏁 METADATA HOÀN TẤT | Time: 2.6 phút


In [21]:
# ═══════════════════════════════════════════════════════════════
# CELL 11: Verification — gộp thành 1 action mỗi dataset
# ═══════════════════════════════════════════════════════════════
from config.minio_config import get_minio_client

print("=" * 70)
print("SILVER LAYER VERIFICATION")
print("=" * 70)

# ── Reviews: 1 action duy nhất thay vì 5 ──
df_rev = spark.read.parquet(SILVER_REVIEWS_PATH)

stats = df_rev.agg(
    F.count('*').alias('total'),
    F.min('rating').alias('min_rating'),
    F.max('rating').alias('max_rating'),
    F.mean('rating').alias('avg_rating'),
    F.count(F.when(F.col('parent_asin').isNull(), 1)).alias('null_parent_asin'),
    F.count(F.when(F.col('user_id').isNull(),     1)).alias('null_user_id'),
    F.count(F.when(F.col('rating').isNull(),      1)).alias('null_rating'),
    F.count(F.when(F.col('review_ts').isNull(),   1)).alias('null_review_ts'),
    F.count(F.when(F.col('review_year') == 2023,  1)).alias('count_2023'),
    F.countDistinct('review_year').alias('distinct_years'),
).collect()[0]

rev_count = stats['total']

print(f"\n Silver Reviews: {rev_count:,} records")
print(f"   Rating:      min={stats['min_rating']:.1f} | "
      f"max={stats['max_rating']:.1f} | avg={stats['avg_rating']:.2f}")
print(f"   Null check:  parent_asin={stats['null_parent_asin']} | "
      f"user_id={stats['null_user_id']} | "
      f"rating={stats['null_rating']} | "
      f"review_ts={stats['null_review_ts']}")
print(f"   Year 2023:   {stats['count_2023']:,} records")
print(f"   Years span:  {stats['distinct_years']} distinct years")

# Schema (không tốn scan)
print(f"\n   Schema:")
df_rev.printSchema()

# Year distribution — 1 action
print(f"\n   Distribution by year:")
df_rev.groupBy('review_year').count().orderBy('review_year').show(30)

# ── File size analysis ──
print(f"\n    File analysis:")
try:
    client        = get_minio_client()
    bucket        = 'electronics-silver'
    objects       = list(client.list_objects(bucket, prefix='reviews/', recursive=True))
    parquet_files = [o for o in objects if o.object_name.endswith('.parquet')]
    if parquet_files:
        sizes_mb = [o.size / (1024**2) for o in parquet_files]
        avg_mb   = sum(sizes_mb) / len(sizes_mb)
        print(f"   Files: {len(parquet_files)} | "
              f"Min: {min(sizes_mb):.1f}MB | "
              f"Max: {max(sizes_mb):.1f}MB | "
              f"Avg: {avg_mb:.1f}MB | "
              f"Total: {sum(sizes_mb)/1024:.2f}GB")
        print(f"   → {'✅ Optimal' if 32 <= avg_mb <= 256 else '⚠️ Consider compaction'}")
except Exception as e:
    print(f"   ⚠️ {e}")

# ── Metadata: 1 action ──
df_meta = spark.read.parquet(SILVER_META_PATH)

meta_stats = df_meta.agg(
    F.count('*').alias('total'),
    F.count('price_numeric').alias('has_price'),
    (F.count('price_numeric') / F.count('*') * 100).alias('price_pct'),
    F.min('price_numeric').alias('min_price'),
    F.max('price_numeric').alias('max_price'),
    F.mean('price_numeric').alias('avg_price'),
).collect()[0]

meta_count = meta_stats['total']
print(f"\n Silver Metadata: {meta_count:,} records")
print(f"   Price coverage: {meta_stats['has_price']:,} "
      f"({meta_stats['price_pct']:.1f}%) | "
      f"range: ${meta_stats['min_price']:.2f} – ${meta_stats['max_price']:.2f} | "
      f"avg: ${meta_stats['avg_price']:.2f}")

print(f"\n{'='*70}")
print(f" SUMMARY:")
print(f"   Silver Reviews: {rev_count:,} | Silver Meta: {meta_count:,}")
print(f"   Pipeline: INPUT SPLIT → MAP → SHUFFLE → REDUCE → OUTPUT")
print(f"{'='*70}")

SILVER LAYER VERIFICATION

 Silver Reviews: 43,408,907 records
   Rating:      min=1.0 | max=5.0 | avg=4.10
   Null check:  parent_asin=0 | user_id=0 | rating=0 | review_ts=0
   Year 2023:   1,884,788 records
   Years span:  27 distinct years

   Schema:
root
 |-- parent_asin: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- rating: float (nullable = true)
 |-- title_clean: string (nullable = true)
 |-- text_clean: string (nullable = true)
 |-- text_word_count: integer (nullable = true)
 |-- review_ts: timestamp (nullable = true)
 |-- helpful_vote: integer (nullable = true)
 |-- verified_purchase: boolean (nullable = true)
 |-- has_text: boolean (nullable = true)
 |-- review_year: integer (nullable = true)
 |-- review_month: integer (nullable = true)


   Distribution by year:
+-----------+-------+
|review_year|  count|
+-----------+-------+
|       1996|      1|
|       1998|      8|
|       1999|    412|
|       2000|   3696|
|       2001|   6754|
|       2002|   

In [22]:
# ═══════════════════════════════════════════════════════════════
# CELL 12: (Tùy chọn) Xóa Bronze chunks sau khi Silver verified
# ═══════════════════════════════════════════════════════════════
from config.minio_config import get_minio_client

def cleanup_bronze():
    """Xóa Bronze chunks sau khi Silver đã verify OK → giải phóng disk."""
    client = get_minio_client()
    bucket = 'electronics-bronze'

    if not client.bucket_exists(bucket):
        print(f"Bucket {bucket} không tồn tại.")
        return

    objects = list(client.list_objects(bucket, recursive=True))
    total_size = sum(obj.size for obj in objects if obj.size)

    print(f"⚠️  Sẽ xóa {len(objects)} objects ({total_size / 1024**3:.2f} GB) từ {bucket}")
    confirm = input("Nhập 'YES' để xác nhận: ")

    if confirm == 'YES':
        from minio.deleteobjects import DeleteObject
        delete_list = [DeleteObject(obj.object_name)
                       for obj in client.list_objects(bucket, recursive=True)]
        errors = list(client.remove_objects(bucket, delete_list))
        if errors:
            print(f"❌ Errors: {errors}")
        else:
            print(f"✅ Đã xóa {len(delete_list)} objects → freed ~{total_size / 1024**3:.2f} GB")
    else:
        print("Đã hủy.")

# Uncomment khi Silver đã verify OK:
cleanup_bronze()

⚠️  Sẽ xóa 153 objects (9.01 GB) từ electronics-bronze


Nhập 'YES' để xác nhận:  YES


✅ Đã xóa 153 objects → freed ~9.01 GB
